**LABEL PROPAGATION** <br>
**Reference**: Near linear time algorithm to detect community structures in large-scale networks (Raghavan et al., 2007) <br>

**igraph doc**: https://igraph.org/python/doc/api/igraph.Graph.html#community_label_propagation


**STEP 0: LOAD LIBRARIES & DATA**

In [ ]:

import pandas as pd
#import json
from pandas import json_normalize
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
#import collections

from networkx.drawing.layout import bipartite_layout
from networkx.algorithms import bipartite
#import networkx as nx
import igraph as ig
#import os
import random

In [ ]:
# df_final = pd.read_csv('/content/drive/MyDrive/MOTIFS PAPER/DATASETS/communities_label_prop.csv')
# df_final = df_final.drop(columns=['Unnamed: 0'])
# df_final

In [ ]:
#Dataset: full edgelist, df with communities
df_communities = pd.read_csv('row_communities_Leid.csv')
df_communities = df_communities.drop(columns=['Unnamed: 0'])

df_edge = pd.read_csv('edgelist_full_final.csv')

#Find weights
#df_edge['count'] = df_edge.groupby(df_edge.columns.tolist(), dropna=False)[df_edge.columns[0]].transform('size')
df_edge['count'] = df_edge.groupby(['user_id_str', 'retweet_author_id'], sort=False)['user_id_str'].transform('size')

df_edge = df_edge.drop_duplicates()
weights = df_edge['count'].to_list()

df_edge = df_edge.drop('count', axis=1)

#Generate graph
edgelist = df_edge.values.tolist()
edges_row = [[str(x) for x in group] for group in edgelist] #convert nodes ID to string, otherwise igraph wont work

vertices = set()
for line in edges_row:
    vertices.update(line)
vertices = sorted(vertices)

#generate empty graph
g_row = ig.Graph()

# add vertices to the graph
g_row.add_vertices(vertices)

# add edges to the graph
g_row.add_edges(edges_row)

In [ ]:
df_vertices = pd.DataFrame(vertices)
df_vertices = df_vertices.rename(columns={0: 'Node ID'})
df_vertices['Node ID'] = df_vertices['Node ID'].astype(int)

In [ ]:
df_vertices

In [ ]:
df_communities[['Node ID', 'Community']]

In [ ]:
#Dataframe with all nodes in order and the respective community label (-1 means no label)
df = df_vertices.merge(df_communities[['Node ID', 'Community']], how='left', on='Node ID')
df = df.fillna(-1)
df['Community'] = df['Community'].astype(int)
df = df.rename(columns={'Community': 'verified_label'})

**STEP 1: LABEL PROPAGATION** <be>


*   **Initial**: a list containing the initial vertex labels. Labels are identified by integers from zero to n − 1 where n is the number of vertices. Negative numbers may also be present in this vector, they represent unlabeled vertices.
*   **Fixed**: a list of booleans for each vertex. True corresponds to vertices whose labeling should not change during the algorithm. It only makes sense if initial labels are also given.



In [ ]:
#Obtaining parameters for label prop algorithm
initial = df['verified_label'].tolist()

fixed = []
for x in initial:
  if x != -1:
    fixed.append(True)
  else:
    fixed.append(False)

In [ ]:
#Label propagation
label_prop = g_row.community_label_propagation(weights=weights, initial= initial, fixed= fixed)

In [ ]:
# Create the pandas DataFrame
df_final= pd.DataFrame(vertices, columns = ['Node ID'])
df_final['Community'] = label_prop.membership
df_final

In [ ]:
# Create the pandas DataFrame
df_final= pd.DataFrame(vertices, columns = ['Node ID'])
df_final['Community'] = label_prop.membership
df_final

In [ ]:
# pd.set_option('display.max_rows', None)
# df_final['Community'].value_counts()

In [ ]:
#Test code: uncomment only for testing
# i = 11
# test_lst = df_communities[df_communities['Community'] == i]['Node ID'].values.tolist()

# for x  in test_lst:
#   node_x = df_final[df_final['Node ID'] == str(x)]['Community']
#   print(node_x)

In [ ]:
#Download both datasets in csv format
df_final.to_csv('communities_label_prop_Leid.csv')

In [ ]:
pd.set_option('display.max_rows', 25)
df_final[df_final['Community'] == 0]